# Exercise 2 — Custom JSON Deserialization

Build a robust custom `JSONDecoder` that reverses the serialization format from Exercise 1 and reconstructs `Stock`, `Trade`, `Decimal`, `date`, and `datetime` instances.

### Design goals
- Decode only explicitly whitelisted types.
- Validate tagged object shape and schema version before reconstruction.
- Preserve exact `Decimal` values.
- Support Python 3.6-compatible ISO date/time parsing.
- Fail loudly on malformed or unsupported tagged objects.
- Provide an optional lenient mode for forward compatibility.
- Verify correctness with end-to-end round-trip tests.


## 1. Domain classes and Exercise 1 serialization contract

This notebook is self-contained. The encoder is included only as supporting code so the decoder can be tested end-to-end.


In [1]:
from datetime import date, datetime
from decimal import Decimal, InvalidOperation
import json
from typing import Any, Dict, Iterable, Optional


TYPE_FIELD = "__type__"
VERSION_FIELD = "__version__"
SCHEMA_VERSION = 1


class Stock:
    def __init__(self, symbol, date, open_, high, low, close, volume):
        self.symbol = symbol
        self.date = date
        self.open = open_
        self.high = high
        self.low = low
        self.close = close
        self.volume = volume

    def __repr__(self):
        return (
            f"Stock(symbol={self.symbol!r}, date={self.date!r}, "
            f"open={self.open!r}, high={self.high!r}, low={self.low!r}, "
            f"close={self.close!r}, volume={self.volume!r})"
        )

    def __eq__(self, other):
        if not isinstance(other, Stock):
            return NotImplemented

        return (
            self.symbol == other.symbol
            and self.date == other.date
            and self.open == other.open
            and self.high == other.high
            and self.low == other.low
            and self.close == other.close
            and self.volume == other.volume
        )


class Trade:
    def __init__(self, symbol, timestamp, order, price, volume, commission):
        self.symbol = symbol
        self.timestamp = timestamp
        self.order = order
        self.price = price
        self.commission = commission
        self.volume = volume

    def __repr__(self):
        return (
            f"Trade(symbol={self.symbol!r}, timestamp={self.timestamp!r}, "
            f"order={self.order!r}, price={self.price!r}, "
            f"volume={self.volume!r}, commission={self.commission!r})"
        )

    def __eq__(self, other):
        if not isinstance(other, Trade):
            return NotImplemented

        return (
            self.symbol == other.symbol
            and self.timestamp == other.timestamp
            and self.order == other.order
            and self.price == other.price
            and self.volume == other.volume
            and self.commission == other.commission
        )


### Supporting encoder

This is the same serialization contract used in Exercise 1. It is included here so this notebook can be executed independently.


In [2]:
class FinancialJSONEncoder(json.JSONEncoder):
    """Encoder compatible with the Exercise 1 solution."""

    def default(self, obj: Any) -> Any:
        # datetime is a subclass of date, so it must be checked first.
        if isinstance(obj, datetime):
            return {
                TYPE_FIELD: "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, date):
            return {
                TYPE_FIELD: "date",
                "value": obj.isoformat(),
            }

        if isinstance(obj, Decimal):
            return {
                TYPE_FIELD: "Decimal",
                "value": str(obj),
            }

        if isinstance(obj, Stock):
            return {
                TYPE_FIELD: "Stock",
                VERSION_FIELD: SCHEMA_VERSION,
                "symbol": obj.symbol,
                "date": obj.date,
                "open": obj.open,
                "high": obj.high,
                "low": obj.low,
                "close": obj.close,
                "volume": obj.volume,
            }

        if isinstance(obj, Trade):
            return {
                TYPE_FIELD: "Trade",
                VERSION_FIELD: SCHEMA_VERSION,
                "symbol": obj.symbol,
                "timestamp": obj.timestamp,
                "order": obj.order,
                "price": obj.price,
                "volume": obj.volume,
                "commission": obj.commission,
            }

        return super().default(obj)


def serialize(data: Any, indent: Optional[int] = 2) -> str:
    """Serialize supported financial-domain objects."""
    return json.dumps(
        data,
        cls=FinancialJSONEncoder,
        indent=indent,
        ensure_ascii=False,
        sort_keys=True,
    )


### Sample data


In [3]:
activity = {
    "quotes": [
        Stock(
            "TSLA",
            date(2018, 11, 22),
            Decimal("338.19"),
            Decimal("338.64"),
            Decimal("337.60"),
            Decimal("338.19"),
            365_607,
        ),
        Stock(
            "AAPL",
            date(2018, 11, 22),
            Decimal("176.66"),
            Decimal("177.25"),
            Decimal("176.64"),
            Decimal("176.78"),
            3_699_184,
        ),
        Stock(
            "MSFT",
            date(2018, 11, 22),
            Decimal("103.25"),
            Decimal("103.48"),
            Decimal("103.07"),
            Decimal("103.11"),
            4_493_689,
        ),
    ],
    "trades": [
        Trade(
            "TSLA",
            datetime(2018, 11, 22, 10, 5, 12),
            "buy",
            Decimal("338.25"),
            100,
            Decimal("9.99"),
        ),
        Trade(
            "AAPL",
            datetime(2018, 11, 22, 10, 30, 5),
            "sell",
            Decimal("177.01"),
            20,
            Decimal("9.99"),
        ),
    ],
}

serialized_activity = serialize(activity)
print(serialized_activity)


{
  "quotes": [
    {
      "__type__": "Stock",
      "__version__": 1,
      "close": {
        "__type__": "Decimal",
        "value": "338.19"
      },
      "date": {
        "__type__": "date",
        "value": "2018-11-22"
      },
      "high": {
        "__type__": "Decimal",
        "value": "338.64"
      },
      "low": {
        "__type__": "Decimal",
        "value": "337.60"
      },
      "open": {
        "__type__": "Decimal",
        "value": "338.19"
      },
      "symbol": "TSLA",
      "volume": 365607
    },
    {
      "__type__": "Stock",
      "__version__": 1,
      "close": {
        "__type__": "Decimal",
        "value": "176.78"
      },
      "date": {
        "__type__": "date",
        "value": "2018-11-22"
      },
      "high": {
        "__type__": "Decimal",
        "value": "177.25"
      },
      "low": {
        "__type__": "Decimal",
        "value": "176.64"
      },
      "open": {
        "__type__": "Decimal",
        "value": "176.66"
   

## 2. Decoder strategy

`json` calls `object_hook` from the inside out. This is exactly what we want: nested tagged `Decimal`, `date`, and `datetime` dictionaries are converted first, and then the outer `Stock` or `Trade` dictionary is reconstructed with already-decoded field values.

The decoder uses a **whitelist** rather than dynamically importing classes based on JSON metadata. This is safer because untrusted JSON cannot request arbitrary Python object construction.


## 3. Validation helpers

Before constructing domain objects, the decoder validates the tagged object's schema. This catches corrupted or unexpectedly changed payloads close to the serialization boundary.


In [4]:
class FinancialJSONDecodeError(ValueError):
    """Raised when tagged financial JSON violates the expected schema."""


def _expect_exact_keys(
    obj: Dict[str, Any],
    expected: Iterable[str],
    type_name: str,
) -> None:
    expected_keys = set(expected)
    actual_keys = set(obj)

    missing = expected_keys - actual_keys
    unexpected = actual_keys - expected_keys

    if missing or unexpected:
        details = []

        if missing:
            details.append("missing={}".format(sorted(missing)))

        if unexpected:
            details.append("unexpected={}".format(sorted(unexpected)))

        raise FinancialJSONDecodeError(
            "Invalid {} payload: {}".format(
                type_name,
                ", ".join(details),
            )
        )


def _expect_instance(
    value: Any,
    expected_type: type,
    field_name: str,
    owner_type: str,
) -> None:
    if not isinstance(value, expected_type):
        raise FinancialJSONDecodeError(
            "{}.{} must be {}, got {}".format(
                owner_type,
                field_name,
                expected_type.__name__,
                type(value).__name__,
            )
        )


def _expect_plain_int(
    value: Any,
    field_name: str,
    owner_type: str,
) -> None:
    # bool is a subclass of int, but is not a meaningful trade volume.
    if not isinstance(value, int) or isinstance(value, bool):
        raise FinancialJSONDecodeError(
            "{}.{} must be int, got {}".format(
                owner_type,
                field_name,
                type(value).__name__,
            )
        )


def _validate_version(
    obj: Dict[str, Any],
    type_name: str,
) -> None:
    version = obj.get(VERSION_FIELD)

    if version != SCHEMA_VERSION:
        raise FinancialJSONDecodeError(
            "Unsupported {} schema version: {!r}; expected {!r}".format(
                type_name,
                version,
                SCHEMA_VERSION,
            )
        )


## 4. Python 3.6-compatible ISO parsers

`date.fromisoformat()` and `datetime.fromisoformat()` were added after Python 3.6, so this solution uses `strptime` while still accepting the ISO strings produced by `date.isoformat()` and `datetime.isoformat()`.

Timezone offsets such as `+03:00` are normalized to `+0300` for compatibility.


In [5]:
def _parse_iso_date(value: Any) -> date:
    if not isinstance(value, str):
        raise FinancialJSONDecodeError(
            "date.value must be str, got {}".format(
                type(value).__name__
            )
        )

    try:
        return datetime.strptime(value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise FinancialJSONDecodeError(
            "Invalid ISO date value: {!r}".format(value)
        ) from exc


def _normalize_timezone_offset(value: str) -> str:
    # Convert a trailing ISO offset such as +03:00 or -05:30 to
    # +0300 / -0530 for broad Python 3.6 strptime compatibility.
    if len(value) >= 6 and value[-6] in "+-" and value[-3] == ":":
        return value[:-3] + value[-2:]

    # Also accept the common UTC designator Z.
    if value.endswith("Z"):
        return value[:-1] + "+0000"

    return value


def _parse_iso_datetime(value: Any) -> datetime:
    if not isinstance(value, str):
        raise FinancialJSONDecodeError(
            "datetime.value must be str, got {}".format(
                type(value).__name__
            )
        )

    normalized = _normalize_timezone_offset(value)

    formats = (
        "%Y-%m-%dT%H:%M:%S.%f%z",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%dT%H:%M:%S",
    )

    for format_ in formats:
        try:
            return datetime.strptime(normalized, format_)
        except ValueError:
            pass

    raise FinancialJSONDecodeError(
        "Invalid ISO datetime value: {!r}".format(value)
    )


## 5. Custom `JSONDecoder`

The core implementation uses `object_hook`. Because nested dictionaries are processed first, `Stock.date`, `Stock.open`, `Trade.timestamp`, etc. have already been converted into their Python types when the containing domain object is reconstructed.

**Strict mode** is the default. If a tagged type is unknown, decoding fails instead of silently guessing.

For forward-compatible pipelines, `strict_types=False` keeps unknown tagged dictionaries untouched while still decoding all known types.


In [6]:
class FinancialJSONDecoder(json.JSONDecoder):
    """Deserialize the tagged JSON representation from Exercise 1."""

    def __init__(self, *args, **kwargs):
        self.strict_types = kwargs.pop("strict_types", True)

        # Force this decoder to use its validated object hook.
        kwargs["object_hook"] = self._object_hook

        super().__init__(*args, **kwargs)

    def _object_hook(self, obj: Dict[str, Any]) -> Any:
        type_name = obj.get(TYPE_FIELD)

        # A normal JSON dictionary needs no special handling.
        if type_name is None:
            return obj

        # ------------------------------------------------------------
        # Decimal
        # ------------------------------------------------------------
        if type_name == "Decimal":
            _expect_exact_keys(
                obj,
                {TYPE_FIELD, "value"},
                "Decimal",
            )

            value = obj["value"]

            if not isinstance(value, str):
                raise FinancialJSONDecodeError(
                    "Decimal.value must be str, got {}".format(
                        type(value).__name__
                    )
                )

            try:
                return Decimal(value)
            except (InvalidOperation, ValueError) as exc:
                raise FinancialJSONDecodeError(
                    "Invalid Decimal value: {!r}".format(value)
                ) from exc

        # ------------------------------------------------------------
        # date
        # ------------------------------------------------------------
        if type_name == "date":
            _expect_exact_keys(
                obj,
                {TYPE_FIELD, "value"},
                "date",
            )

            return _parse_iso_date(obj["value"])

        # ------------------------------------------------------------
        # datetime
        # ------------------------------------------------------------
        if type_name == "datetime":
            _expect_exact_keys(
                obj,
                {TYPE_FIELD, "value"},
                "datetime",
            )

            return _parse_iso_datetime(obj["value"])

        # ------------------------------------------------------------
        # Stock
        # ------------------------------------------------------------
        if type_name == "Stock":
            _expect_exact_keys(
                obj,
                {
                    TYPE_FIELD,
                    VERSION_FIELD,
                    "symbol",
                    "date",
                    "open",
                    "high",
                    "low",
                    "close",
                    "volume",
                },
                "Stock",
            )

            _validate_version(obj, "Stock")

            _expect_instance(
                obj["symbol"],
                str,
                "symbol",
                "Stock",
            )

            # datetime is also an instance of date, so explicitly
            # exclude it here.
            if not isinstance(obj["date"], date) or isinstance(
                obj["date"], datetime
            ):
                raise FinancialJSONDecodeError(
                    "Stock.date must be date, got {}".format(
                        type(obj["date"]).__name__
                    )
                )

            for field_name in ("open", "high", "low", "close"):
                _expect_instance(
                    obj[field_name],
                    Decimal,
                    field_name,
                    "Stock",
                )

            _expect_plain_int(
                obj["volume"],
                "volume",
                "Stock",
            )

            return Stock(
                symbol=obj["symbol"],
                date=obj["date"],
                open_=obj["open"],
                high=obj["high"],
                low=obj["low"],
                close=obj["close"],
                volume=obj["volume"],
            )

        # ------------------------------------------------------------
        # Trade
        # ------------------------------------------------------------
        if type_name == "Trade":
            _expect_exact_keys(
                obj,
                {
                    TYPE_FIELD,
                    VERSION_FIELD,
                    "symbol",
                    "timestamp",
                    "order",
                    "price",
                    "volume",
                    "commission",
                },
                "Trade",
            )

            _validate_version(obj, "Trade")

            _expect_instance(
                obj["symbol"],
                str,
                "symbol",
                "Trade",
            )

            _expect_instance(
                obj["timestamp"],
                datetime,
                "timestamp",
                "Trade",
            )

            _expect_instance(
                obj["order"],
                str,
                "order",
                "Trade",
            )

            _expect_instance(
                obj["price"],
                Decimal,
                "price",
                "Trade",
            )

            _expect_instance(
                obj["commission"],
                Decimal,
                "commission",
                "Trade",
            )

            _expect_plain_int(
                obj["volume"],
                "volume",
                "Trade",
            )

            return Trade(
                symbol=obj["symbol"],
                timestamp=obj["timestamp"],
                order=obj["order"],
                price=obj["price"],
                volume=obj["volume"],
                commission=obj["commission"],
            )

        # Tagged objects are never dynamically imported or instantiated.
        if self.strict_types:
            raise FinancialJSONDecodeError(
                "Unsupported tagged object type: {!r}".format(
                    type_name
                )
            )

        # Optional forward-compatible behavior: preserve unknown tagged
        # dictionaries as ordinary dictionaries.
        return obj


## 6. Public deserialization helper

Application code should not need to repeatedly know which decoder class or configuration to supply.


In [7]:
def deserialize(
    payload: str,
    strict_types: bool = True,
) -> Any:
    """Deserialize JSON produced by FinancialJSONEncoder."""
    if not isinstance(payload, str):
        raise TypeError(
            "payload must be str, got {}".format(
                type(payload).__name__
            )
        )

    return json.loads(
        payload,
        cls=FinancialJSONDecoder,
        strict_types=strict_types,
    )


## 7. Decode the serialized activity


In [8]:
decoded_activity = deserialize(serialized_activity)

print(decoded_activity)
print()

print(
    "First quote type:",
    type(decoded_activity["quotes"][0]).__name__,
)

print(
    "First quote date type:",
    type(decoded_activity["quotes"][0].date).__name__,
)

print(
    "First quote open type:",
    type(decoded_activity["quotes"][0].open).__name__,
)

print(
    "First trade type:",
    type(decoded_activity["trades"][0]).__name__,
)

print(
    "First trade timestamp type:",
    type(decoded_activity["trades"][0].timestamp).__name__,
)


{'quotes': [Stock(symbol='TSLA', date=datetime.date(2018, 11, 22), open=Decimal('338.19'), high=Decimal('338.64'), low=Decimal('337.60'), close=Decimal('338.19'), volume=365607), Stock(symbol='AAPL', date=datetime.date(2018, 11, 22), open=Decimal('176.66'), high=Decimal('177.25'), low=Decimal('176.64'), close=Decimal('176.78'), volume=3699184), Stock(symbol='MSFT', date=datetime.date(2018, 11, 22), open=Decimal('103.25'), high=Decimal('103.48'), low=Decimal('103.07'), close=Decimal('103.11'), volume=4493689)], 'trades': [Trade(symbol='TSLA', timestamp=datetime.datetime(2018, 11, 22, 10, 5, 12), order='buy', price=Decimal('338.25'), volume=100, commission=Decimal('9.99')), Trade(symbol='AAPL', timestamp=datetime.datetime(2018, 11, 22, 10, 30, 5), order='sell', price=Decimal('177.01'), volume=20, commission=Decimal('9.99'))]}

First quote type: Stock
First quote date type: date
First quote open type: Decimal
First trade type: Trade
First trade timestamp type: datetime


## 8. Round-trip verification

A correct encoder/decoder pair should satisfy:

```text
original Python objects
        ↓
       JSON
        ↓
reconstructed Python objects
```

without losing domain types or decimal precision.


In [9]:
assert decoded_activity == activity

first_quote = decoded_activity["quotes"][0]
first_trade = decoded_activity["trades"][0]

# Stock reconstruction.
assert isinstance(first_quote, Stock)
assert isinstance(first_quote.date, date)
assert not isinstance(first_quote.date, datetime)
assert isinstance(first_quote.open, Decimal)
assert isinstance(first_quote.high, Decimal)
assert isinstance(first_quote.low, Decimal)
assert isinstance(first_quote.close, Decimal)
assert first_quote.open == Decimal("338.19")
assert first_quote.volume == 365_607

# Trade reconstruction.
assert isinstance(first_trade, Trade)
assert isinstance(first_trade.timestamp, datetime)
assert isinstance(first_trade.price, Decimal)
assert isinstance(first_trade.commission, Decimal)
assert first_trade.price == Decimal("338.25")
assert first_trade.commission == Decimal("9.99")
assert first_trade.volume == 100

print("Round-trip verification passed.")


Round-trip verification passed.


## 9. Verify every reconstructed object


In [10]:
assert all(
    isinstance(stock, Stock)
    for stock in decoded_activity["quotes"]
)

assert all(
    isinstance(stock.date, date)
    and not isinstance(stock.date, datetime)
    for stock in decoded_activity["quotes"]
)

assert all(
    isinstance(value, Decimal)
    for stock in decoded_activity["quotes"]
    for value in (
        stock.open,
        stock.high,
        stock.low,
        stock.close,
    )
)

assert all(
    isinstance(trade, Trade)
    for trade in decoded_activity["trades"]
)

assert all(
    isinstance(trade.timestamp, datetime)
    for trade in decoded_activity["trades"]
)

assert all(
    isinstance(trade.price, Decimal)
    and isinstance(trade.commission, Decimal)
    for trade in decoded_activity["trades"]
)

print("All reconstructed domain/value types are correct.")


All reconstructed domain/value types are correct.


## 10. Precision preservation test

The key reason not to serialize `Decimal` as JSON floating-point numbers is precision. This test uses a value that should never be routed through a binary `float`.


In [11]:
precision_payload = {
    "value": Decimal("0.123456789012345678901234567890"),
}

precision_json = serialize(precision_payload)
precision_result = deserialize(precision_json)

assert precision_result["value"] == precision_payload["value"]
assert isinstance(precision_result["value"], Decimal)

print("Exact Decimal round-trip passed:")
print(precision_result["value"])


Exact Decimal round-trip passed:
0.123456789012345678901234567890


## 11. Datetime precision test

The decoder also preserves microseconds from `datetime.isoformat()`.


In [12]:
datetime_payload = {
    "timestamp": datetime(
        2024,
        5,
        17,
        14,
        22,
        31,
        987654,
    )
}

datetime_json = serialize(datetime_payload)
datetime_result = deserialize(datetime_json)

assert datetime_result["timestamp"] == datetime_payload["timestamp"]
assert datetime_result["timestamp"].microsecond == 987654

print("Datetime microsecond round-trip passed.")


Datetime microsecond round-trip passed.


## 12. Schema-version validation

The `__version__` field prevents the decoder from silently interpreting a future, incompatible representation as the current schema.


In [13]:
bad_version = json.dumps(
    {
        TYPE_FIELD: "Stock",
        VERSION_FIELD: 999,
        "symbol": "TSLA",
        "date": {
            TYPE_FIELD: "date",
            "value": "2018-11-22",
        },
        "open": {
            TYPE_FIELD: "Decimal",
            "value": "1.00",
        },
        "high": {
            TYPE_FIELD: "Decimal",
            "value": "1.00",
        },
        "low": {
            TYPE_FIELD: "Decimal",
            "value": "1.00",
        },
        "close": {
            TYPE_FIELD: "Decimal",
            "value": "1.00",
        },
        "volume": 1,
    }
)

try:
    deserialize(bad_version)
except FinancialJSONDecodeError as exc:
    print("Version check correctly rejected payload:")
    print(exc)


Version check correctly rejected payload:
Unsupported Stock schema version: 999; expected 1


## 13. Unknown-type handling

Dynamic class loading based on a JSON field would be dangerous and unnecessarily complex. The decoder instead maintains an explicit whitelist.

Unknown tagged types are rejected in strict mode and preserved as dictionaries in optional lenient mode.


In [14]:
future_payload = json.dumps(
    {
        TYPE_FIELD: "FutureInstrument",
        VERSION_FIELD: 1,
        "symbol": "XYZ",
    }
)

# Strict mode is the safe default.
try:
    deserialize(future_payload)
except FinancialJSONDecodeError as exc:
    print("Strict type check correctly rejected payload:")
    print(exc)


# Lenient mode is useful when a pipeline needs to preserve data created
# by a newer producer without understanding that data yet.
future_object = deserialize(
    future_payload,
    strict_types=False,
)

assert isinstance(future_object, dict)
assert future_object[TYPE_FIELD] == "FutureInstrument"
assert future_object["symbol"] == "XYZ"

print()
print("Lenient mode preserved the unknown object:")
print(future_object)


Strict type check correctly rejected payload:
Unsupported tagged object type: 'FutureInstrument'

Lenient mode preserved the unknown object:
{'__type__': 'FutureInstrument', '__version__': 1, 'symbol': 'XYZ'}


## 14. Malformed payload checks

Tagged data is treated as a small serialization protocol, so malformed protocol objects are rejected explicitly.


In [15]:
malformed_decimal = json.dumps(
    {
        TYPE_FIELD: "Decimal",
        "value": ["not", "a", "string"],
    }
)

try:
    deserialize(malformed_decimal)
except FinancialJSONDecodeError as exc:
    print("Malformed Decimal correctly rejected:")
    print(exc)


Malformed Decimal correctly rejected:
Decimal.value must be str, got list


In [16]:
malformed_stock = json.dumps(
    {
        TYPE_FIELD: "Stock",
        VERSION_FIELD: SCHEMA_VERSION,
        "symbol": "TSLA",
        # Remaining required fields are intentionally omitted.
    }
)

try:
    deserialize(malformed_stock)
except FinancialJSONDecodeError as exc:
    print("Malformed Stock correctly rejected:")
    print(exc)


Malformed Stock correctly rejected:
Invalid Stock payload: missing=['close', 'date', 'high', 'low', 'open', 'volume']


In [17]:
wrong_volume = json.dumps(
    {
        TYPE_FIELD: "Trade",
        VERSION_FIELD: SCHEMA_VERSION,
        "symbol": "TSLA",
        "timestamp": {
            TYPE_FIELD: "datetime",
            "value": "2018-11-22T10:05:12",
        },
        "order": "buy",
        "price": {
            TYPE_FIELD: "Decimal",
            "value": "338.25",
        },
        "volume": "100",
        "commission": {
            TYPE_FIELD: "Decimal",
            "value": "9.99",
        },
    }
)

try:
    deserialize(wrong_volume)
except FinancialJSONDecodeError as exc:
    print("Incorrect field type correctly rejected:")
    print(exc)


Incorrect field type correctly rejected:
Trade.volume must be int, got str


## 15. Compact application-facing API

A small `dumps` / `loads` API keeps JSON configuration out of business logic and makes the serialization implementation replaceable later.


In [18]:
def dumps(data: Any, **kwargs) -> str:
    """Serialize financial-domain data with sensible defaults."""
    options = {
        "cls": FinancialJSONEncoder,
        "ensure_ascii": False,
        "sort_keys": True,
    }

    options.update(kwargs)
    return json.dumps(data, **options)


def loads(
    payload: str,
    strict_types: bool = True,
    **kwargs
) -> Any:
    """Deserialize financial-domain data with sensible defaults."""
    options = {
        "cls": FinancialJSONDecoder,
        "strict_types": strict_types,
    }

    options.update(kwargs)
    return json.loads(payload, **options)


In [19]:
# Compact JSON can be useful for storage/network transmission.
compact_payload = dumps(
    activity,
    separators=(",", ":"),
)

compact_result = loads(compact_payload)

assert compact_result == activity

print(
    "Compact payload size:",
    len(compact_payload),
    "characters",
)
print("Convenience API round-trip passed.")


Compact payload size: 1430 characters
Convenience API round-trip passed.


## 16. Final integration test

This final test exercises the complete serialization contract in one operation.


In [20]:
def round_trip(value: Any) -> Any:
    """Serialize and immediately deserialize a supported value."""
    return loads(dumps(value))


result = round_trip(activity)

assert result == activity
assert result is not activity
assert result["quotes"][0] is not activity["quotes"][0]
assert result["trades"][0] is not activity["trades"][0]

print("=" * 60)
print("ALL EXERCISE 2 TESTS PASSED")
print("=" * 60)


ALL EXERCISE 2 TESTS PASSED


## Result

`FinancialJSONDecoder` now reverses the Exercise 1 representation safely and losslessly:

- `Decimal` values remain exact.
- `date` and `datetime` values regain their original Python types.
- `Stock` and `Trade` dictionaries become domain objects again.
- Nested values are reconstructed naturally through `object_hook`.
- Schema versions and payload shapes are validated.
- Unknown tagged types are never dynamically imported or instantiated.
- Strict mode provides fail-fast behavior.
- Optional lenient mode supports forward-compatible pipelines.
- Round-trip tests demonstrate that serialization and deserialization preserve the original object graph's values and types.

This completes **Exercise 2**. Marshmallow is intentionally not introduced here because that is the subject of **Exercise 3**.
